# Pre-deforestation


In [1]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
# ============================================================
# 0. Imports
# ============================================================
import numpy as np
import pandas as pd
import xarray as xr
import json
from pathlib import Path
import matplotlib.pyplot as plt

from hbv_bmi import HBV_Bmi


# ============================================================
# 1. Paths and periods
# ============================================================
data_dir = Path("./Data")

precip_nc = data_dir / "manning_ERA5_precip_daily.nc"
evap_nc   = data_dir / "manning_ERA5_evap_daily.nc"
discharge_file = data_dir / "5202080_Q_Day.Cmd.txt"

shape_area = 6642 * 1e6  # m²

exp_start = "2014-01-01"
exp_end   = "2025-01-01"

pre_start  = "2014-01-01"
pre_end    = "2019-10-01"

post_start = "2020-03-01"
post_end   = "2025-01-01"


# ============================================================
# 2. Load discharge and clean
# ============================================================
obs = pd.read_csv(
    discharge_file,
    delimiter=';',
    skiprows=36,
    header=0,
    encoding='cp1252'
)
obs.columns = ["Date", "Time", "Q"]
obs["Q"] = pd.to_numeric(obs["Q"], errors="coerce")
obs = obs.dropna(subset=["Q"])
obs["Date"] = pd.to_datetime(obs["Date"])
obs = obs.set_index("Date").drop(columns=["Time"]).sort_index()

# m³/s → mm/day if needed
if obs["Q"].max() > 50:
    obs["Q"] = obs["Q"] * 86400 / shape_area * 1000.0

obs.loc[obs["Q"] > 4000, "Q"] = np.nan
obs.loc[obs["Q"] < 0, "Q"] = np.nan
obs = obs.loc[exp_start:exp_end]


# ============================================================
# 3. Precompute time index from forcing (no per-step datetime)
# ============================================================
ds_precip = xr.open_dataset(precip_nc)
time_index = pd.to_datetime(ds_precip["time"].values)
ds_precip.close()

mask_exp = (time_index >= pd.to_datetime(exp_start)) & (time_index <= pd.to_datetime(exp_end))
time_index = time_index[mask_exp]

# Align obs to this time index
obs = obs.reindex(time_index)


# ============================================================
# 4. Objective function (NSE, logNSE, De)
# ============================================================
def calibration_objective(model_df, obs_df, start_cal, end_cal):
    hydro = pd.concat(
        [model_df.reindex(obs_df.index, method="ffill"), obs_df],
        axis=1
    )
    hydro = hydro.loc[start_cal:end_cal]

    sim = hydro["model"].values.copy()
    obs = hydro["Q"].values.copy()

    # Simple interpolation for NaNs in obs
    for i in range(1, len(obs) - 1):
        if np.isnan(obs[i]) and not np.isnan(obs[i - 1]) and not np.isnan(obs[i + 1]):
            obs[i] = 0.5 * (obs[i - 1] + obs[i + 1])

    mask = ~np.isnan(obs)
    sim = sim[mask]
    obs = obs[mask]

    if len(obs) < 3:
        return np.nan, np.nan, np.nan

    sim_clipped = np.clip(sim, 1e-6, None)
    obs_clipped = np.clip(obs, 1e-6, None)

    nse = 1.0 - np.sum((sim - obs) ** 2) / np.sum((obs - obs.mean()) ** 2)
    log_nse = 1.0 - np.sum((np.log(sim_clipped) - np.log(obs_clipped)) ** 2) / \
                    np.sum((np.log(obs_clipped) - np.log(obs_clipped).mean()) ** 2)
    De = np.sqrt((1.0 - nse) ** 2 + (1.0 - log_nse) ** 2)
    return nse, log_nse, De


# ============================================================
# 5. Ensemble setup
# ============================================================
p_min_initial = np.array([0., 0.4,  40., 1.0, 0.001, 1.,  0.01, 0.0001])
p_max_initial = np.array([8., 0.8, 800., 2.5, 0.3,   3.,  0.1,  0.01])

N = 20  # reduced for speed
parameters = np.zeros((8, N))

np.random.seed(6)
for i in range(8):
    parameters[i, :] = np.random.uniform(p_min_initial[i], p_max_initial[i], N)


# ============================================================
# 6. One reusable JSON config file
# ============================================================
config_path = Path("./hbv_config.json")

base_config = {
    "precipitation_file": str(precip_nc),
    "potential_evaporation_file": str(evap_nc),
    "initial_storage": "0,100,0,5",
    "parameters": ""  # overwritten each iteration
}

with open(config_path, "w") as f:
    json.dump(base_config, f)


# ============================================================
# 7. Run ensemble (HBV_Bmi, optimized usage)
# ============================================================
objectives = []

for n in range(N):
    par = parameters[:, n]

    # Update parameters and overwrite same small JSON file
    base_config["parameters"] = ",".join(str(p) for p in par)
    with open(config_path, "w") as f:
        json.dump(base_config, f)

    model = HBV_Bmi()
    model.initialize(str(config_path))

    n_steps = model.end_timestep
    Q_m = np.zeros(n_steps, dtype=float)

    # Fast loop: only update + read Q, no datetime/xarray/list appends
    for t in range(n_steps):
        model.update()
        Q_m[t] = model.Q

    model_time = pd.date_range(exp_start, periods=n_steps, freq="D")
    model_df = pd.DataFrame({"model": Q_m}, index=model_time)
    model_df = model_df.loc[exp_start:exp_end]

    nse, log_nse, De = calibration_objective(
        model_df,
        obs[["Q"]],
        pre_start,
        pre_end
    )
    objectives.append((nse, log_nse, De))


# ============================================================
# 8. Best parameters
# ============================================================
objectives = np.array(objectives)
De_vals = objectives[:, 2]
best_idx = np.nanargmin(De_vals)
best_par = parameters[:, best_idx]

print("Best ensemble member index:", best_idx)
print("Best parameters (Imax, Ce, Sumax, beta, Pmax, Tlag, Kf, Ks):")
print(best_par)

SUmax_pre = best_par[2]
print("SUmax pre-deforestation =", SUmax_pre, "mm")


# ============================================================
# 9. Run HBV_Bmi with best parameters over full period
# ============================================================
base_config["parameters"] = ",".join(str(p) for p in best_par)
with open(config_path, "w") as f:
    json.dump(base_config, f)

model = HBV_Bmi()
model.initialize(str(config_path))

n_steps = model.end_timestep
Q_best = np.zeros(n_steps, dtype=float)

for t in range(n_steps):
    model.update()
    Q_best[t] = model.Q

model_time = pd.date_range(exp_start, periods=n_steps, freq="D")
sim_full = pd.DataFrame({"model": Q_best}, index=model_time)
sim_full = sim_full.loc[exp_start:exp_end]

obs_df = obs[["Q"]]


# ============================================================
# 10. Metrics: pre and post
# ============================================================
def print_metrics(label, start, end):
    sim = sim_full.loc[start:end]["model"]
    ob = obs_df.loc[start:end]["Q"]
    nse, lognse, De = calibration_objective(
        sim.to_frame("model"),
        ob.to_frame("Q"),
        start,
        end
    )
    print(f"\n{label}")
    print("  NSE    =", nse)
    print("  logNSE =", lognse)
    print("  De     =", De)
    return nse, lognse, De

pre_nse, pre_lognse, pre_De = print_metrics("Pre-deforestation (2014–2019-10-01):", pre_start, pre_end)
post_nse, post_lognse, post_De = print_metrics("Post-deforestation (2020–2025-01-01):", post_start, post_end)


# ============================================================
# 11. Plots
# ============================================================
def plot_period(title, start, end, color):
    sim = sim_full.loc[start:end]["model"]
    ob = obs_df.loc[start:end]["Q"]
    plt.figure(figsize=(12,4))
    plt.plot(ob.index, ob.values, label="Obs", color="k")
    plt.plot(sim.index, sim.values, label="HBV_Bmi", color=color)
    plt.title(title)
    plt.ylabel("Q [mm/day]")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_period("Pre-deforestation (2014–2019-10-01)", pre_start, pre_end, "tab:blue")
plot_period("Post-deforestation (2020–2025-01-01)", post_start, post_end, "tab:orange")


ModuleNotFoundError: No module named 'bmipy'

POST DEFORESTATION

In [ ]:

# Post-deforestation (2020-03-01 → 2025-01-01)
post2_start = "2020-03-01"
post2_end   = "2025-01-01"


post2_sim = sim_full.loc[post2_start:post2_end]
post2_obs = obs.loc[post2_start:post2_end][["Q"]]

post2_nse, post2_lognse, post2_De = calibration_objective(
    post2_sim, post2_obs, post2_start, post2_end
)

print("\nPost-deforestation (extended) metrics:")
print("  NSE    =", post2_nse)
print("  logNSE =", post2_lognse)
print("  De     =", post2_De)

# Plot
plt.figure(figsize=(12,4))
plt.plot(post2_obs.index, post2_obs["Q"], label="Obs (post-extended)", color="k")
plt.plot(post2_sim.index, post2_sim["model"], label="HBV (post-extended)", color="tab:red")
plt.title("Post-deforestation (2020–2025): modeled vs observed")
plt.ylabel("Q [mm/day]")
plt.legend()
plt.tight_layout()
plt.show()
